In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/DERİNMODEL")

RAW_DATA_DIR = PROJECT_DIR / "01_veri_seti" / "ham_veri"
PROCESSED_DATA_DIR = PROJECT_DIR / "01_veri_seti" / "islenmis_veri"
NOTEBOOK_DIR = PROJECT_DIR / "02_notebooklar"

OUTPUT_DIR = PROJECT_DIR / "03_ciktilar"
GRAPH_DIR = OUTPUT_DIR / "grafikler"
TABLE_DIR = OUTPUT_DIR / "tablolar"
SAMPLE_DIR = OUTPUT_DIR / "ornek_gorseller"
MODEL_SUMMARY_DIR = OUTPUT_DIR / "model_ozetleri"

MODEL_DIR = PROJECT_DIR / "04_modeller"

PREPROCESS_GRAPH_DIR = GRAPH_DIR / "2_on_isleme"
PREPROCESS_TABLE_DIR = TABLE_DIR

for path in [
    PROCESSED_DATA_DIR,
    PREPROCESS_GRAPH_DIR,
    PREPROCESS_TABLE_DIR,
    MODEL_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

print("02 veri ön işleme klasörleri hazır.")
print("Proje klasörü:", PROJECT_DIR)

02 veri ön işleme klasörleri hazır.
Proje klasörü: /content/drive/MyDrive/DERİNMODEL


In [8]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/DERİNMODEL")

RAW_DATA_DIR = PROJECT_DIR / "01_veri_seti" / "ham_veri"
DATASET_ROOT = RAW_DATA_DIR / "dagm_2007" / "DAGM_KaggleUpload"

OUTPUT_DIR = PROJECT_DIR / "03_ciktilar"
TABLE_DIR = OUTPUT_DIR / "tablolar"
GRAPH_DIR = OUTPUT_DIR / "grafikler"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset root var mı?:", DATASET_ROOT.exists())
print("Dataset root:", DATASET_ROOT)

class_names = ["Class1", "Class2", "Class3", "Class4", "Class5", "Class6"]

records = []

for class_name in class_names:
    for split in ["Train", "Test"]:
        split_dir = DATASET_ROOT / class_name / split
        label_dir = split_dir / "Label"

        image_files = sorted([
            p for p in split_dir.glob("*.PNG")
            if p.is_file()
        ])

        label_files = set()
        if label_dir.exists():
            label_files = set([p.name for p in label_dir.glob("*_label.PNG")])

        for image_path in image_files:
            image_name = image_path.name
            image_stem = image_path.stem

            label_name = f"{image_stem}_label.PNG"
            label_path = label_dir / label_name

            is_defective = label_name in label_files

            records.append({
                "class": class_name,
                "split": split,
                "dosya_adi": image_name,
                "image_path": str(image_path),
                "label_dosyasi": label_name if is_defective else "",
                "label_path": str(label_path) if is_defective else "",
                "hedef": 1 if is_defective else 0,
                "hedef_adi": "kusurlu" if is_defective else "kusursuz"
            })

manifest_df = pd.DataFrame(records)

manifest_path = TABLE_DIR / "dagm_2007_binary_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("Manifest yeniden oluşturuldu ve kaydedildi:")
print(manifest_path)

print("\nToplam kayıt:", len(manifest_df))

print("\nSplit dağılımı:")
print(manifest_df["split"].value_counts())

print("\nHedef dağılımı:")
print(manifest_df["hedef_adi"].value_counts())

print("\nClass bazında hedef dağılımı:")
display(pd.crosstab(manifest_df["class"], manifest_df["hedef_adi"]))

manifest_df.head()

Dataset root var mı?: True
Dataset root: /content/drive/MyDrive/DERİNMODEL/01_veri_seti/ham_veri/dagm_2007/DAGM_KaggleUpload
Manifest yeniden oluşturuldu ve kaydedildi:
/content/drive/MyDrive/DERİNMODEL/03_ciktilar/tablolar/dagm_2007_binary_manifest.csv

Toplam kayıt: 6900

Split dağılımı:
split
Train    3450
Test     3450
Name: count, dtype: int64

Hedef dağılımı:
hedef_adi
kusursuz    6000
kusurlu      900
Name: count, dtype: int64

Class bazında hedef dağılımı:


hedef_adi,kusurlu,kusursuz
class,,
Class1,150,1000
Class2,150,1000
Class3,150,1000
Class4,150,1000
Class5,150,1000
Class6,150,1000


,class,split,dosya_adi,image_path,label_dosyasi,label_path,hedef,hedef_adi
0,Class1,Train,0576.PNG,/content/drive/MyDrive/DERİNMODEL/01_veri_seti...,,,0,kusursuz
1,Class1,Train,0577.PNG,/content/drive/MyDrive/DERİNMODEL/01_veri_seti...,,,0,kusursuz
2,Class1,Train,0578.PNG,/content/drive/MyDrive/DERİNMODEL/01_veri_seti...,,,0,kusursuz
3,Class1,Train,0579.PNG,/content/drive/MyDrive/DERİNMODEL/01_veri_seti...,,,0,kusursuz
4,Class1,Train,0580.PNG,/content/drive/MyDrive/DERİNMODEL/01_veri_seti...,,,0,kusursuz


In [9]:
from sklearn.model_selection import train_test_split

# Sadece Train verisini alıyoruz
train_full_df = manifest_df[manifest_df["split"] == "Train"].copy()

# Kaggle Test verisini aynen koruyoruz
test_df = manifest_df[manifest_df["split"] == "Test"].copy()

# Validation sadece Train içinden ayrılacak
train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.20,
    random_state=42,
    stratify=train_full_df[["class", "hedef"]]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["veri_bolumu"] = "train"
val_df["veri_bolumu"] = "validation"
test_df["veri_bolumu"] = "test"

split_manifest_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

split_manifest_path = TABLE_DIR / "dagm_2007_train_validation_test_manifest.csv"
split_manifest_df.to_csv(split_manifest_path, index=False)

print("Train / Validation / Test manifest kaydedildi:")
print(split_manifest_path)

print("\nVeri bölümü dağılımı:")
print(split_manifest_df["veri_bolumu"].value_counts())

print("\nVeri bölümü - hedef dağılımı:")
display(pd.crosstab(split_manifest_df["veri_bolumu"], split_manifest_df["hedef_adi"]))

print("\nVeri bölümü - class - hedef dağılımı:")
display(pd.crosstab(
    [split_manifest_df["veri_bolumu"], split_manifest_df["class"]],
    split_manifest_df["hedef_adi"]
))

Train / Validation / Test manifest kaydedildi:
/content/drive/MyDrive/DERİNMODEL/03_ciktilar/tablolar/dagm_2007_train_validation_test_manifest.csv

Veri bölümü dağılımı:
veri_bolumu
test          3450
train         2760
validation     690
Name: count, dtype: int64

Veri bölümü - hedef dağılımı:


hedef_adi,kusurlu,kusursuz
veri_bolumu,,
test,454,2996
train,357,2403
validation,89,601



Veri bölümü - class - hedef dağılımı:


hedef_adi           kusurlu  kusursuz
veri_bolumu class                    
test        Class1       71       504
            Class2       84       491
            Class3       84       491
            Class4       68       507
            Class5       80       495
            Class6       67       508
train       Class1       63       397
            Class2       53       407
            Class3       53       407
            Class4       66       394
            Class5       56       404
            Class6       66       394
validation  Class1       16        99
            Class2       13       102
            Class3       13       102
            Class4       16        99
            Class5       14       101
            Class6       17        98

In [10]:
import shutil
from pathlib import Path

# İşlenmiş veri klasörü
PROCESSED_DATA_DIR = PROJECT_DIR / "01_veri_seti" / "islenmis_veri"

# Eski işlenmiş veri varsa sil
if PROCESSED_DATA_DIR.exists():
    shutil.rmtree(PROCESSED_DATA_DIR)

# Yeni klasör yapısını oluştur
for split in ["train", "validation", "test"]:
    for target_name in ["kusurlu", "kusursuz"]:
        folder_path = PROCESSED_DATA_DIR / split / target_name
        folder_path.mkdir(parents=True, exist_ok=True)

print("İşlenmiş veri klasör yapısı oluşturuldu:")
print(PROCESSED_DATA_DIR)

İşlenmiş veri klasör yapısı oluşturuldu:
/content/drive/MyDrive/DERİNMODEL/01_veri_seti/islenmis_veri


In [11]:
def copy_images_to_processed_dataset(df, split_name):
    copied_count = 0

    for _, row in df.iterrows():
        source_path = Path(row["image_path"])
        target_class = row["hedef_adi"]  # kusurlu veya kusursuz

        # Aynı dosya adları farklı classlarda olabileceği için benzersiz isim veriyoruz
        new_file_name = f'{row["class"]}_{row["split"]}_{row["dosya_adi"]}'

        target_path = PROCESSED_DATA_DIR / split_name / target_class / new_file_name

        shutil.copy2(source_path, target_path)
        copied_count += 1

    print(f"{split_name} klasörüne kopyalanan görsel sayısı:", copied_count)


copy_images_to_processed_dataset(train_df, "train")
copy_images_to_processed_dataset(val_df, "validation")
copy_images_to_processed_dataset(test_df, "test")

print("\nTüm görseller işlenmiş veri klasörüne kopyalandı.")

train klasörüne kopyalanan görsel sayısı: 2760
validation klasörüne kopyalanan görsel sayısı: 690
test klasörüne kopyalanan görsel sayısı: 3450

Tüm görseller işlenmiş veri klasörüne kopyalandı.


In [12]:
for split in ["train", "validation", "test"]:
    print(f"\n{split.upper()}")

    total = 0
    for target_name in ["kusurlu", "kusursuz"]:
        folder_path = PROCESSED_DATA_DIR / split / target_name
        count = len(list(folder_path.glob("*.PNG")))
        total += count
        print(target_name, ":", count)

    print("Toplam:", total)


TRAIN
kusurlu : 357
kusursuz : 2403
Toplam: 2760

VALIDATION
kusurlu : 89
kusursuz : 601
Toplam: 690

TEST
kusurlu : 454
kusursuz : 2996
Toplam: 3450


In [13]:
import pandas as pd

preprocess_summary = pd.DataFrame([
    {
        "kontrol": "Kullanılan sınıflar",
        "sonuc": "Class1-Class6"
    },
    {
        "kontrol": "Sınıflandırma tipi",
        "sonuc": "Binary classification: kusurlu / kusursuz"
    },
    {
        "kontrol": "Train görsel sayısı",
        "sonuc": 2760
    },
    {
        "kontrol": "Validation görsel sayısı",
        "sonuc": 690
    },
    {
        "kontrol": "Test görsel sayısı",
        "sonuc": 3450
    },
    {
        "kontrol": "Train kusurlu / kusursuz",
        "sonuc": "357 / 2403"
    },
    {
        "kontrol": "Validation kusurlu / kusursuz",
        "sonuc": "89 / 601"
    },
    {
        "kontrol": "Test kusurlu / kusursuz",
        "sonuc": "454 / 2996"
    },
    {
        "kontrol": "Hazır test seti korundu mu?",
        "sonuc": "Evet, orijinal Test bölümü test seti olarak ayrıldı"
    },
    {
        "kontrol": "Validation nasıl oluşturuldu?",
        "sonuc": "Sadece orijinal Train verisi içinden ayrıldı"
    },
    {
        "kontrol": "Etiketleme yöntemi",
        "sonuc": "Label klasöründe maske dosyası olan görseller kusurlu, olmayanlar kusursuz kabul edildi"
    },
    {
        "kontrol": "Veri sızıntısı önlemi",
        "sonuc": "Test verisi eğitim ve validation içine dahil edilmedi"
    },
    {
        "kontrol": "Model için klasör yapısı",
        "sonuc": "train / validation / test altında kusurlu ve kusursuz klasörleri oluşturuldu"
    }
])

preprocess_summary_path = PREPROCESS_TABLE_DIR / "dagm_2007_veri_on_isleme_final_ozet.csv"
preprocess_summary.to_csv(preprocess_summary_path, index=False)

print("Veri ön işleme final özeti kaydedildi:")
print(preprocess_summary_path)

preprocess_summary

Veri ön işleme final özeti kaydedildi:
/content/drive/MyDrive/DERİNMODEL/03_ciktilar/tablolar/dagm_2007_veri_on_isleme_final_ozet.csv


,kontrol,sonuc
0,Kullanılan sınıflar,Class1-Class6
1,Sınıflandırma tipi,Binary classification: kusurlu / kusursuz
2,Train görsel sayısı,2760
3,Validation görsel sayısı,690
4,Test görsel sayısı,3450
5,Train kusurlu / kusursuz,357 / 2403
6,Validation kusurlu / kusursuz,89 / 601
7,Test kusurlu / kusursuz,454 / 2996
8,Hazır test seti korundu mu?,"Evet, orijinal Test bölümü test seti olarak ay..."
9,Validation nasıl oluşturuldu?,Sadece orijinal Train verisi içinden ayrıldı
